# Kaggle 2/2 — Entrenamiento y experimentos

**Configuración (panel derecho → *Session options*):**
- **Accelerator:** GPU T4 x2 (se usa una) o GPU P100.
- **Internet:** On.
- **Inputs:**
  1. La salida del notebook 1: *Add Input* → *Your Work* → notebook de extracción.
  2. *(Desde la segunda corrida)* La salida de la **versión anterior de este mismo notebook**, para continuar donde quedó.

**Cómo correrlo:** *Save Version* → **Save & Run All (Commit)**.

Kaggle corta cada sesión a las 12 h, y la GPU tiene una cuota de ~30 h por semana. Por eso el notebook trabaja con un **presupuesto de tiempo**: entrena los experimentos en orden, y cuando se acerca al límite pausa el que está en curso y termina limpio. La siguiente versión, con la anterior como input, salta lo terminado y continúa el pausado.

Todo queda en *Output*: `runs/` (checkpoints), `results/` (métricas), `resultados.csv` y figuras de Grad-CAM.

In [ ]:
import os, subprocess
REPO = 'https://github.com/Julianchu850/proyecto-redes-neuronales-deteccion-deepfake.git'
if not os.path.exists('/tmp/repo'):
    subprocess.run(['git', 'clone', '-q', REPO, '/tmp/repo'], check=True)
else:
    subprocess.run(['git', '-C', '/tmp/repo', 'pull', '-q'], check=True)
%cd /tmp/repo
!git log --oneline -1

## Configuración

In [ ]:
PRESUPUESTO_HORAS = 11      # tiempo de esta sesión (Kaggle corta a las 12 h)
EPOCAS = 20                 # con early stopping (paciencia 5)
BATCH = 64                  # bajar a 32 si aparece 'CUDA out of memory'
MAS_E3 = True               # incluir el experimento E3 (técnica no vista) al final de la cola

# Orden de la cola: primero lo indispensable para el informe
EXPERIMENTOS = [
    # (nombre del run, modelo, argumentos extra)
    ('meso4',            'meso4',            ''),
    ('dos_ramas',        'dos_ramas',        ''),
    ('efficientnet_b0',  'efficientnet_b0',  '--lr 3e-4'),
    ('solo_espacial',    'solo_espacial',    ''),
    ('solo_frecuencia',  'solo_frecuencia',  ''),
    ('dos_ramas_sin_se', 'dos_ramas_sin_se', ''),
    ('dos_ramas_srm',    'dos_ramas_srm',    ''),
]
TECNICAS = ['Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures']
if MAS_E3:
    for modelo in ['dos_ramas', 'solo_espacial', 'meso4']:
        for fuera in TECNICAS:
            resto = ' '.join(t for t in TECNICAS if t != fuera)
            EXPERIMENTOS.append((f'e3/{modelo}_sin_{fuera}', modelo, f'--metodos {resto}'))
print(len(EXPERIMENTOS), 'experimentos en cola')

## Datos y estado previo

In [ ]:
import glob, os, shutil, time
T_INICIO = time.time()
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

zips = glob.glob('/kaggle/input/**/faces_ffpp.zip', recursive=True)
assert zips, 'No encontré faces_ffpp.zip: agrega como input la salida del notebook 1'
if not os.path.exists('/tmp/data/faces/ffpp'):
    !mkdir -p /tmp/data/faces && unzip -q "{zips[0]}" -d /tmp/data/faces/
splits = glob.glob('/kaggle/input/**/splits/ffpp/train.csv', recursive=True)
SPLITS = os.path.dirname(splits[0])
print('Rostros:', zips[0], '| Splits:', SPLITS)

# Continuar desde la versión anterior de este notebook (si se agregó como input)
W = '/kaggle/working'
for previo in glob.glob('/kaggle/input/**/runs', recursive=True):
    if glob.glob(f'{previo}/**/ultimo.pt', recursive=True):
        print('Recuperando estado previo de', previo)
        shutil.copytree(previo, f'{W}/runs', dirs_exist_ok=True)
        resultados_previos = os.path.join(os.path.dirname(previo), 'results')
        if os.path.exists(resultados_previos):
            shutil.copytree(resultados_previos, f'{W}/results', dirs_exist_ok=True)
        break
os.makedirs(f'{W}/runs', exist_ok=True); os.makedirs(f'{W}/results', exist_ok=True)

## Cola de experimentos

In [ ]:
import pandas as pd
DATOS = '/tmp/data/faces/ffpp'
hay_faceshifter = 'FaceShifter' in set(pd.read_csv(f'{SPLITS}/test.csv')['metodo'])
WORKERS = min(os.cpu_count(), 4)
FALLIDOS = []

def minutos_restantes():
    return PRESUPUESTO_HORAS * 60 - (time.time() - T_INICIO) / 60

for nombre, modelo, extra in EXPERIMENTOS:
    run, res = f'{W}/runs/{nombre}', f'{W}/results/{nombre}'
    if os.path.exists(f'{res}/metricas.json'):
        print(f'✔ {nombre}'); continue
    margen = minutos_restantes() - 15   # reserva para evaluar y guardar
    if margen < 20:
        print(f'⏸ Sin tiempo para {nombre}: queda para la próxima versión.'); break
    print(f'▶ {nombre} (quedan {margen:.0f} min)')
    !python -m src.entrenar --modelo {modelo} --datos {DATOS} --splits {SPLITS} --salida {run} \
        --epocas {EPOCAS} --batch {BATCH} --workers {WORKERS} --reanudar --max-minutos {margen} {extra}
    if os.path.exists(f'{run}/PAUSADO'):
        print('⏸ Presupuesto agotado: la próxima versión continúa este experimento.'); break
    if not os.path.exists(f'{run}/TERMINADO'):
        # El entrenamiento se cayó (error o falta de memoria): no se evalúa un modelo a medias
        print(f'✗ {nombre} falló: revisa el error arriba. La próxima versión lo reintenta desde su último checkpoint.')
        FALLIDOS.append(nombre); continue
    !python -m src.evaluar --checkpoint {run}/mejor.pt --datos {DATOS} --csv {SPLITS}/test.csv \
        --salida {res} --workers {WORKERS}
    if hay_faceshifter and not nombre.startswith('e3/'):
        !python -m src.evaluar --checkpoint {run}/mejor.pt --datos {DATOS} --csv {SPLITS}/test.csv \
            --salida {res}_faceshifter --workers {WORKERS} --metodos FaceShifter
    # ultimo.pt (con el optimizador) ya no hace falta: libera espacio de la salida
    if os.path.exists(f'{run}/TERMINADO'):
        os.remove(f'{run}/ultimo.pt')

## Resultados

In [ ]:
import json
filas = []
for f in sorted(glob.glob(f'{W}/results/**/metricas.json', recursive=True)):
    m = json.load(open(f))
    fila = {'experimento': os.path.relpath(os.path.dirname(f), f'{W}/results'),
            'AUC frame': m['frame']['auc'], 'AUC video': m['video']['auc'],
            'acc video': m['video']['accuracy'], 'F1 video': m['video']['f1']}
    fila.update({f'AUC {k}': v['auc_video'] for k, v in m.get('por_metodo', {}).items()})
    filas.append(fila)
tabla = pd.DataFrame(filas).set_index('experimento').round(4) if filas else pd.DataFrame()
tabla.to_csv(f'{W}/resultados.csv')
tabla

In [ ]:
import matplotlib.pyplot as plt
historiales = {os.path.relpath(os.path.dirname(f), f'{W}/runs'): pd.read_csv(f)
               for f in sorted(glob.glob(f'{W}/runs/*/historial.csv'))}
if historiales:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for n, h in historiales.items():
        ax[0].plot(h.epoca, h.perdida_train, label=n); ax[1].plot(h.epoca, h.auc_val_video, label=n)
    ax[0].set_title('Pérdida de entrenamiento'); ax[1].set_title('AUC de validación (video)')
    for a in ax: a.set_xlabel('época')
    ax[1].legend(fontsize=7); plt.tight_layout(); plt.savefig(f'{W}/curvas.png', dpi=120); plt.show()

## Grad-CAM (E5)

Solo para los modelos ya terminados.

In [ ]:
from IPython.display import Image as Imagen, display
for nombre in ['meso4', 'dos_ramas', 'efficientnet_b0']:
    run, sal = f'{W}/runs/{nombre}', f'{W}/results/gradcam/{nombre}'
    if os.path.exists(f'{run}/TERMINADO') and not os.path.exists(f'{sal}/gradcam_promedio.png'):
        !python -m src.gradcam --checkpoint {run}/mejor.pt --datos {DATOS} --csv {SPLITS}/test.csv --salida {sal}
    if os.path.exists(f'{sal}/gradcam_promedio.png'):
        display(Imagen(f'{sal}/gradcam_promedio.png'))

## Siguiente paso

- Si quedaron experimentos pendientes: *Save Version* de nuevo, agregando como input la salida de esta versión.
- Para el demo, descarga `runs/dos_ramas/mejor.pt` desde *Output* y córrelo en el Mac:
  `python -m app.demo --checkpoint mejor.pt`